# Protocol v3 Paper-Ready Evaluation

This notebook is intentionally compact and focused on defensible core outputs:

1. formal product definition,
2. strict real-data pipeline (Birdeye only),
3. break-even fee-yield computation,
4. strategy ranking by break-even,
5. limitations for paper reporting.

Product constraints:

- `cover = 1`,
- `B = p_l` (barrier equals CL lower bound),
- CL width fixed to `+-7.5%`.


## 1) Formal Definition

At weekly entry spot `S0`, with width `w = 0.075`:

- `p_l = S0(1-w)`
- `p_u = S0(1+w)`
- `B = p_l`

Corridor payout:

$$
\Pi(S_T)=\min\left(Cap,\max\left(0, V(S_0)-V(\max(S_T,B))\right)\right),\quad Cap=V(S_0)-V(B)
$$

Premium rule:

$$
Premium = \max\left(0, FV\cdot m_{vol} - y_{split}\cdot E[Fees]\right)
$$

Protocol break-even condition (joint viability):

- `E[LP return] >= 0` and `E[RT return] >= 0`.

Benchmark break-even condition (LP-only strategies):

- `E[LP return] >= 0`.


In [1]:
import os, json, time
import numpy as np
import pandas as pd
import requests
from scipy.special import ndtr
from numpy.polynomial.hermite import hermgauss
import matplotlib.pyplot as plt
from pathlib import Path

def resolve_data_dir():
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        repo = parent / 'lh-protocol-v3'
        if (repo / 'protocol-src').exists():
            cand = repo / 'notebooks' / 'data'
            cand.mkdir(parents=True, exist_ok=True)
            return str(cand)
    fallback = here / 'data'
    fallback.mkdir(parents=True, exist_ok=True)
    return str(fallback)

DATA_DIR = resolve_data_dir()
rng = np.random.default_rng(42)
print('DATA_DIR:', DATA_DIR)


DATA_DIR: /home/sowelo/Scrivania/LiquidityHedge/lh-protocol-v3/notebooks/data


In [2]:
# 2) Birdeye-only real data loader (hard fail if unavailable)
BIRDEYE_API_KEY = os.getenv('BIRDEYE_API_KEY', '').strip() or 'ed577a4a6a4f480fa659b4f18673e4b1'
SOL_MINT = 'So11111111111111111111111111111111111111112'

def fetch_birdeye_daily_sol(days=1150, chunk_days=90):
    if not BIRDEYE_API_KEY:
        raise RuntimeError('BIRDEYE_API_KEY is missing.')
    now = int(time.time())
    start = now - int(days) * 24 * 3600
    rows = []
    url = 'https://public-api.birdeye.so/defi/ohlcv'
    headers = {'X-API-KEY': BIRDEYE_API_KEY, 'Accept': 'application/json', 'x-chain': 'solana'}
    cur = start
    while cur < now:
        nxt = min(cur + int(chunk_days) * 24 * 3600, now)
        params = {'address': SOL_MINT, 'type': '1D', 'time_from': int(cur), 'time_to': int(nxt)}
        r = requests.get(url, params=params, headers=headers, timeout=20)
        r.raise_for_status()
        js = r.json()
        if not bool(js.get('success', False)):
            raise RuntimeError(f'Birdeye request failed for [{cur}, {nxt}]')
        items = js.get('data', {}).get('items', [])
        if not isinstance(items, list):
            raise RuntimeError('Unexpected Birdeye schema: data.items is not list')
        rows.extend(items)
        cur = nxt
    if len(rows) == 0:
        raise RuntimeError('Birdeye returned empty OHLCV items')
    df = pd.DataFrame(rows)
    if 'unixTime' in df.columns:
        ts = pd.to_datetime(df['unixTime'], unit='s', utc=True)
    elif 'startUnixTime' in df.columns:
        ts = pd.to_datetime(df['startUnixTime'], unit='s', utc=True)
    else:
        raise RuntimeError(f'Unexpected Birdeye columns: {list(df.columns)}')
    if 'c' in df.columns:
        close = pd.to_numeric(df['c'], errors='coerce')
    elif 'close' in df.columns:
        close = pd.to_numeric(df['close'], errors='coerce')
    else:
        raise RuntimeError(f'Unexpected Birdeye close columns: {list(df.columns)}')
    out = pd.DataFrame({'open_time': ts, 'close': close}).dropna().drop_duplicates('open_time').sort_values('open_time').reset_index(drop=True)
    if len(out) < 365:
        raise RuntimeError(f'Birdeye returned too few daily points ({len(out)})')
    return out

hist = fetch_birdeye_daily_sol(days=1150, chunk_days=90)
closes = hist['close'].to_numpy(float)
print('Source: birdeye_SOL_daily')
print('Data points:', len(closes))


Source: birdeye_SOL_daily
Data points: 1150


In [3]:
# 3) Core pricing + simulators (strict in-range fee model)
WIDTH = 0.075
T_WEEK = 7/365
N_LIQ = 10_000.0
PROTOCOL_FEE_RATE = 0.015
QUAD_N = 48
QUAD_NODES, QUAD_WEIGHTS = hermgauss(QUAD_N)
MARKUP_FLOOR = 0.99

def cl_value(S, L, p_l, p_u):
    S = np.asarray(S, float)
    sp, spl, spu = np.sqrt(np.clip(S, 1e-12, None)), np.sqrt(p_l), np.sqrt(p_u)
    below = S <= p_l
    above = S >= p_u
    a = np.where(below, L*(spu-spl)/(spl*spu), np.where(above, 0.0, L*(spu-sp)/(sp*spu)))
    b = np.where(below, 0.0, np.where(above, L*(spu-spl), L*(sp-spl)))
    return a*S + b

def make_pos(S0, lev=1.0):
    p_l, p_u = S0*(1-WIDTH), S0*(1+WIDTH)
    B = p_l
    L = N_LIQ*lev
    V0 = float(cl_value(np.array([S0]), L, p_l, p_u)[0])
    Vb = float(cl_value(np.array([B]), L, p_l, p_u)[0])
    cap = max(0.0, V0 - Vb)
    return {'p_l': p_l, 'p_u': p_u, 'B': B, 'L': L, 'V0': V0, 'cap': cap}

def corridor_payoff(S_T, S0, pos):
    V_eff = cl_value(np.maximum(S_T, pos['B']), pos['L'], pos['p_l'], pos['p_u'])
    raw = np.maximum(0.0, np.where(S_T >= S0, 0.0, pos['V0'] - V_eff))
    return np.minimum(pos['cap'], raw)

def fv_quadrature(S0, sigma, pos, T=T_WEEK):
    S_T = S0*np.exp(-0.5*sigma**2*T + sigma*np.sqrt(T)*QUAD_NODES*np.sqrt(2))
    pay = corridor_payoff(S_T, S0, pos)
    return max(0.0, float(np.sum(QUAD_WEIGHTS*pay)/np.sqrt(np.pi)))

def iv_rv_regime_switching(s7, s30, stress):
    ratio = float(s7) / max(float(s30), 1e-9)
    IVRV_MIN, IVRV_MAX = 0.99, 1.12
    RATIO_LOW, RATIO_HIGH = 0.90, 1.30
    frac = np.clip((ratio - RATIO_LOW)/(RATIO_HIGH - RATIO_LOW), 0.0, 1.0)
    return float(IVRV_MIN + (IVRV_MAX - IVRV_MIN) * frac)

lr_hist = np.diff(np.log(closes))
RV_ANCHOR = float(np.clip(np.std(lr_hist, ddof=1)*np.sqrt(365), 0.10, 3.0))

def build_weekly_blocks(prices, block_days=8):
    prices = np.asarray(prices, float)
    return np.array([prices[i:i+block_days] for i in range(len(prices)-block_days+1)])

def sample_weekly_path(blocks, n_weeks=52, rng_local=None):
    rng_eff = rng_local if rng_local is not None else rng
    idx = rng_eff.integers(0, len(blocks), size=n_weeks)
    return blocks[idx]

blocks = build_weekly_blocks(closes, block_days=8)

def simulate_protocol_mc(y_split, fee_day, n_paths=400, n_weeks=52, lev=1.2, tx_bps_open=2.0, tx_bps_settle=2.0):
    lp_w, rt_w = [], []
    exec_cost_rate = (tx_bps_open + tx_bps_settle)/10_000
    seed = int((round(float(y_split),3)*1e6 + round(float(fee_day),7)*1e6)*1000) % 2**32
    rng_local = np.random.default_rng(seed)
    for _ in range(n_paths):
        rt_cap = 2_000_000.0
        idle_yield_week = (1+0.002)**7 - 1
        for wk in sample_weekly_path(blocks, n_weeks=n_weeks, rng_local=rng_local):
            S0, ST = float(wk[0]), float(wk[-1])
            lr = np.diff(np.log(wk))
            rv30_proxy = float(np.clip(np.std(lr, ddof=1)*np.sqrt(365), 0.25, 1.8))
            pos = make_pos(S0, lev=lev)
            if pos['cap'] <= 1e-9:
                continue
            fv = fv_quadrature(S0, rv30_proxy, pos)
            ivrv = iv_rv_regime_switching(rv30_proxy, RV_ANCHOR, rv30_proxy/max(RV_ANCHOR,1e-9) > 1.25)
            m_vol = max(MARKUP_FLOOR, ivrv)
            t_days = np.arange(1, 8) / 365.0
            sigma = max(rv30_proxy, 1e-9)
            mu = -0.5 * (sigma**2) * t_days
            sd = sigma * np.sqrt(t_days)
            a_log = np.log(1.0 - WIDTH)
            b_log = np.log(1.0 + WIDTH)
            p_in = ndtr((b_log - mu)/sd) - ndtr((a_log - mu)/sd)
            exp_in_range_frac = float(np.mean(p_in))
            exp_fees = pos['V0'] * fee_day * 7 * exp_in_range_frac
            premium = max(0.0, fv*m_vol - y_split*exp_fees)
            realized_fee_day = float(np.clip(rng_local.normal(fee_day,0.0007),0.0003,0.02))
            in_range_frac_real = float(np.mean((wk[1:] >= pos['p_l']) & (wk[1:] <= pos['p_u'])))
            fees = pos['V0'] * realized_fee_day * 7 * in_range_frac_real
            payoff = float(corridor_payoff(np.array([ST]), S0, pos)[0])
            Vend = float(cl_value(np.array([ST]), pos['L'], pos['p_l'], pos['p_u'])[0])
            protocol_exec_cost = exec_cost_rate * pos['V0']
            lp_pnl = (Vend - pos['V0']) + fees*(1-y_split) + payoff - premium - protocol_exec_cost
            rt_pnl = premium*(1-PROTOCOL_FEE_RATE) + fees*y_split - payoff
            lp_ret = lp_pnl/max(pos['V0'],1e-9)
            rt_ret = rt_pnl/max(rt_cap,1e-9)
            rt_cap *= (1 + rt_ret + idle_yield_week)
            lp_w.append(lp_ret); rt_w.append(rt_ret)
    return np.array(lp_w,float), np.array(rt_w,float)

def cl_amount_sol(S, L, p_l, p_u):
    sp, spl, spu = np.sqrt(max(S,1e-12)), np.sqrt(p_l), np.sqrt(p_u)
    if S <= p_l: return L*(spu-spl)/(spl*spu)
    if S >= p_u: return 0.0
    return L*(spu-sp)/(sp*spu)

def simulate_perp_mc(mode='fixed', fee_day=0.0023, n_paths=400, n_weeks=52, lev=1.2, tx_cost_bps=2.0, funding_bps_day=1.0):
    out=[]
    tx_cost = tx_cost_bps/10_000
    funding = funding_bps_day/10_000
    seed = int((round(float(fee_day),7)*1e6)*1000 + (0 if mode=='fixed' else 1)*999) % 2**32
    rng_local = np.random.default_rng(seed)
    for _ in range(n_paths):
        for wk in sample_weekly_path(blocks, n_weeks=n_weeks, rng_local=rng_local):
            S0, ST = float(wk[0]), float(wk[-1])
            pos = make_pos(S0, lev=lev)
            a0 = cl_amount_sol(S0, pos['L'], pos['p_l'], pos['p_u'])
            hedge_qty = 0.5*a0
            decile = (pos['p_u']-pos['p_l'])/10
            anchor = S0
            fee_income = perp_pnl = funding_cf = 0.0
            tcost = abs(hedge_qty)*S0*tx_cost
            realized_fee_day = float(np.clip(rng_local.normal(fee_day,0.0007),0.0003,0.02))
            for d in range(1,len(wk)):
                s_prev, s_cur = float(wk[d-1]), float(wk[d])
                in_range = (s_prev >= pos['p_l']) and (s_prev <= pos['p_u'])
                fee_income += pos['V0'] * realized_fee_day * (1.0 if in_range else 0.0)
                perp_pnl += -hedge_qty*(s_cur-s_prev)
                funding_cf -= abs(hedge_qty*s_prev)*funding
                if mode=='dynamic' and abs(s_cur-anchor)>=decile:
                    drop_frac = max(0.0,(S0-s_cur)/max(pos['p_u']-pos['p_l'],1e-9))
                    target_ratio = float(np.clip(0.5 + drop_frac, 0.1, 1.0))
                    a_cur = cl_amount_sol(s_cur, pos['L'], pos['p_l'], pos['p_u'])
                    target_qty = target_ratio*a_cur
                    dq = target_qty-hedge_qty
                    tcost += abs(dq)*s_cur*tx_cost
                    hedge_qty = target_qty
                    anchor = s_cur
            tcost += abs(hedge_qty)*ST*tx_cost
            Vend = float(cl_value(np.array([ST]), pos['L'], pos['p_l'], pos['p_u'])[0])
            lp_pnl = (Vend-pos['V0']) + fee_income + perp_pnl + funding_cf - tcost
            out.append(lp_pnl/max(pos['V0'],1e-9))
    return np.array(out,float)

def simulate_plain_lp_mc(fee_day=0.0023, n_paths=400, n_weeks=52, lev=1.2):
    out=[]
    seed = int((round(float(fee_day),7)*1e6)*1000) % 2**32
    rng_local = np.random.default_rng(seed)
    for _ in range(n_paths):
        for wk in sample_weekly_path(blocks, n_weeks=n_weeks, rng_local=rng_local):
            S0, ST = float(wk[0]), float(wk[-1])
            pos = make_pos(S0, lev=lev)
            realized_fee_day = float(np.clip(rng_local.normal(fee_day,0.0007),0.0003,0.02))
            in_range_frac_real = float(np.mean((wk[1:] >= pos['p_l']) & (wk[1:] <= pos['p_u'])))
            fees = pos['V0']*realized_fee_day*7*in_range_frac_real
            Vend = float(cl_value(np.array([ST]), pos['L'], pos['p_l'], pos['p_u'])[0])
            out.append(((Vend-pos['V0'])+fees)/max(pos['V0'],1e-9))
    return np.array(out,float)

print('Core simulators ready (strict in-range fee model).')


Core simulators ready (strict in-range fee model).


In [4]:
# 4) Core break-even + ranking outputs
yield_split_grid = np.round(np.array([0.05,0.10,0.15,0.20,0.25,0.30,0.40]), 3)
fee_grid_be = np.round(np.linspace(0.0012, 0.0080, 15), 6)

proto_rows=[]
for ys in yield_split_grid:
    be_joint = np.nan
    for fd in fee_grid_be:
        lp, rt = simulate_protocol_mc(y_split=float(ys), fee_day=float(fd), n_paths=70, n_weeks=28)
        if len(lp)==0 or len(rt)==0:
            continue
        if float(np.mean(lp)) >= 0.0 and float(np.mean(rt)) >= 0.0:
            be_joint = float(fd)
            break
    proto_rows.append({'yield_split': float(ys), 'strategy': 'protocol_lp', 'breakeven_fee_day': be_joint})

def min_fee_for_lp_strategy(sim_fn, fee_grid):
    for fd in fee_grid:
        lp = sim_fn(fee_day=float(fd))
        if len(lp)==0:
            continue
        if float(np.mean(lp)) >= 0.0:
            return float(fd)
    return np.nan

plain_be = min_fee_for_lp_strategy(lambda fee_day: simulate_plain_lp_mc(fee_day=float(fee_day), n_paths=70, n_weeks=28), fee_grid_be)
perp_fixed_be = min_fee_for_lp_strategy(lambda fee_day: simulate_perp_mc(mode='fixed', fee_day=float(fee_day), n_paths=70, n_weeks=28), fee_grid_be)
perp_dynamic_be = min_fee_for_lp_strategy(lambda fee_day: simulate_perp_mc(mode='dynamic', fee_day=float(fee_day), n_paths=70, n_weeks=28), fee_grid_be)

be_core = pd.DataFrame(proto_rows + [
    {'yield_split': np.nan, 'strategy': 'plain_lp', 'breakeven_fee_day': plain_be},
    {'yield_split': np.nan, 'strategy': 'perp_fixed_lp', 'breakeven_fee_day': perp_fixed_be},
    {'yield_split': np.nan, 'strategy': 'perp_dynamic_lp', 'breakeven_fee_day': perp_dynamic_be},
])
be_core.to_csv(os.path.join(DATA_DIR,'v3_paper_strategy_fee_breakeven.csv'), index=False)

proto_feasible = be_core[(be_core['strategy']=='protocol_lp') & (~be_core['breakeven_fee_day'].isna())]
proto_best = float(proto_feasible['breakeven_fee_day'].min()) if len(proto_feasible)>0 else np.nan

rank_core = pd.DataFrame([
    {'strategy':'protocol_lp_best_joint', 'breakeven_fee_day': proto_best},
    {'strategy':'plain_lp', 'breakeven_fee_day': float(plain_be)},
    {'strategy':'perp_fixed_lp', 'breakeven_fee_day': float(perp_fixed_be)},
    {'strategy':'perp_dynamic_lp', 'breakeven_fee_day': float(perp_dynamic_be)},
]).sort_values(['breakeven_fee_day'], ascending=True, na_position='last').reset_index(drop=True)
rank_core['rank'] = np.arange(1, len(rank_core)+1)
rank_core.to_csv(os.path.join(DATA_DIR,'v3_paper_strategy_fee_breakeven_ranking.csv'), index=False)

print('Saved:', os.path.join(DATA_DIR,'v3_paper_strategy_fee_breakeven.csv'))
print('Saved:', os.path.join(DATA_DIR,'v3_paper_strategy_fee_breakeven_ranking.csv'))
print('--- Strategy break-even table ---')
print(be_core.to_string(index=False))
print('--- Core ranking (lower fee_day is better) ---')
rank_core


Saved: /home/sowelo/Scrivania/LiquidityHedge/lh-protocol-v3/notebooks/data/v3_paper_strategy_fee_breakeven.csv
Saved: /home/sowelo/Scrivania/LiquidityHedge/lh-protocol-v3/notebooks/data/v3_paper_strategy_fee_breakeven_ranking.csv
--- Strategy break-even table ---
 yield_split        strategy  breakeven_fee_day
        0.05     protocol_lp           0.005571
        0.10     protocol_lp           0.005571
        0.15     protocol_lp           0.005571
        0.20     protocol_lp           0.005086
        0.25     protocol_lp           0.005571
        0.30     protocol_lp           0.006543
        0.40     protocol_lp           0.005086
         NaN        plain_lp           0.005571
         NaN   perp_fixed_lp           0.005571
         NaN perp_dynamic_lp           0.005571
--- Core ranking (lower fee_day is better) ---


,strategy,breakeven_fee_day,rank
0,protocol_lp_best_joint,0.005086,1
1,plain_lp,0.005571,2
2,perp_fixed_lp,0.005571,3
3,perp_dynamic_lp,0.005571,4


## 5) Conclusions and Limitations

Primary paper-facing outputs:

- `v3_paper_strategy_fee_breakeven.csv`
- `v3_paper_strategy_fee_breakeven_ranking.csv`

Interpretation:

- lower `breakeven_fee_day` means a strategy needs less LP fee-yield to be non-negative.
- protocol feasibility is stricter: LP and RT must both have non-negative mean returns.

Limitations to report:

1. Perpetual costs are assumed constants (`2 bps` trade cost, `1 bp/day` funding), not time-series venue data.
2. Break-even thresholds are approximate on a finite `fee_day` scan grid and finite Monte Carlo path count.
3. Conclusions are conditional on this bootstrap regime and model structure.
